# Section 0: Environment Setup

This section installs the required dependencies, checks for GPU availability, sets deterministic seeds for reproducibility, and defines a global configuration dictionary.  The code also mounts Google Drive to ensure that outputs are persistently stored.

In [ ]:

# Install dependencies if they are missing.  Import modules to check versions.
import importlib, subprocess, sys, os, random, json

# Define the packages needed for this pipeline.  These are installed only if not already present.
packages = [
    "mediapipe",
    "ultralytics",
    "opencv-python-headless",
    "numpy",
    "pandas",
    "plotly",
    "scipy",
    "pillow",
    "pytesseract",
    "scikit-image",
    "tqdm"
]
for pkg in packages:
    try:
        importlib.import_module(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# Version checks and GPU availability
import mediapipe as mp
import cv2
import numpy as np
import pandas as pd
import torch
import plotly
import scipy
from PIL import Image

print("MediaPipe version:", mp.__version__)
print("Ultralytics version:", importlib.import_module('ultralytics').__version__)
print("OpenCV version:", cv2.__version__)
print("PyTorch CUDA available:", torch.cuda.is_available())

# Deterministic seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if hasattr(torch, 'manual_seed'):
    torch.manual_seed(SEED)

# Global configuration dictionary
CONFIG = {
    "fps_target": 120,
    "frame_resize": (640, 360),
    "hand_conf_thresh": 0.5,
    "ball_conf_thresh": 0.25,
    "smoothing_window": 7,
    "output_dir": "/content/drive/MyDrive/pitch_analysis_output",
    "ocr_enabled": False,
    "quick_test_frames": 150
}

# Ensure the output directory exists
os.makedirs(CONFIG["output_dir"], exist_ok=True)

# Mount Google Drive for persistent storage
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print("Google Drive could not be mounted. Running in local mode:", e)


# Section 1: Data Ingestion

This section handles interactive video upload, extracts basic metadata such as frames per second (fps), resolution, and duration, and provides utilities for sampling frames with stride control.  An optional optical character recognition (OCR) routine can parse on‑screen pitch metrics, though it is disabled by default.

In [ ]:

from google.colab import files
import cv2
import numpy as np
import pandas as pd
import os
import pytesseract
from PIL import Image
from typing import List, Tuple, Dict


def upload_videos() -> Dict[str, str]:
    """Upload video files using the Colab file picker and save them locally."""
    uploaded = files.upload()
    video_paths: Dict[str, str] = {}
    for name, data in uploaded.items():
        path = os.path.join('/content', name)
        with open(path, 'wb') as f:
            f.write(data)
        video_paths[name] = path
    return video_paths


def extract_metadata(video_path: str) -> Dict[str, float]:
    """Return metadata including fps, width, height, duration, and frame count."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    duration = frame_count / fps if fps > 0 else 0
    cap.release()
    return {
        "fps": fps,
        "width": width,
        "height": height,
        "duration_sec": duration,
        "frame_count": int(frame_count)
    }


def sample_frames(video_path: str, stride: int = 30, resize: Tuple[int, int] = None, max_frames: int = 150) -> List[np.ndarray]:
    """Extract frames from a video at regular intervals.  Optionally resize frames and limit the total number extracted."""
    frames: List[np.ndarray] = []
    cap = cv2.VideoCapture(video_path)
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % stride == 0:
            if resize:
                frame = cv2.resize(frame, resize)
            frames.append(frame)
            if len(frames) >= max_frames:
                break
        idx += 1
    cap.release()
    return frames


def save_sample_frames(frames: List[np.ndarray], prefix: str, out_dir: str) -> None:
    """Save sampled frames into a QA directory for quick inspection."""
    qa_dir = os.path.join(out_dir, f"{prefix}_qa_frames")
    os.makedirs(qa_dir, exist_ok=True)
    for i, frame in enumerate(frames):
        cv2.imwrite(os.path.join(qa_dir, f"frame_{i:04d}.jpg"), frame)


def parse_ocr_overlay(frame: np.ndarray) -> Dict[str, str]:
    """Perform OCR on a single frame if OCR is enabled in the global config."""
    if not CONFIG.get("ocr_enabled", False):
        return {}
    text = pytesseract.image_to_string(Image.fromarray(frame))
    return {"overlay_text": text.strip()}

# Upload videos
VIDEO_PATHS = upload_videos()
print("Uploaded videos:", VIDEO_PATHS)

# Extract and display metadata
VIDEO_METADATA = {name: extract_metadata(path) for name, path in VIDEO_PATHS.items()}
print(pd.DataFrame(VIDEO_METADATA).T)


# Section 2: Hand Landmark Pipeline

Use MediaPipe's hand landmarker to detect the throwing hand in each frame.  Finger tip landmarks (thumb, index, middle, ring, pinky) are tracked, smoothed using a moving average, and exported to CSV and JSON.  A QA video overlay visualises the detected landmarks.

In [ ]:

import os
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions
from typing import Tuple, Dict, List


class HandLandmarkPipeline:
    """Pipeline for detecting and smoothing hand landmarks using MediaPipe."""
    def __init__(self, config: Dict):
        self.config = config
        # Model file path
        self.model_path = 'hand_landmarker.task'
        # Download model if missing
        if not os.path.exists(self.model_path):
            import urllib.request
            url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker_lite/float16/1/hand_landmarker_lite.task'
            print('Downloading MediaPipe hand landmarker model...')
            urllib.request.urlretrieve(url, self.model_path)
        options = HandLandmarkerOptions(
            base_options=vision.BaseOptions(model_asset_path=self.model_path),
            num_hands=1,
            min_hand_detection_confidence=self.config.get('hand_conf_thresh', 0.5),
            min_hand_presence_confidence=self.config.get('hand_conf_thresh', 0.5),
            min_tracking_confidence=self.config.get('hand_conf_thresh', 0.5)
        )
        self.landmarker = HandLandmarker.create_from_options(options)
        self.smoothing_window = self.config.get('smoothing_window', 7)

    def process_video(self, video_path: str, quick_test: bool = False) -> Tuple[pd.DataFrame, str]:
        """Extract hand landmarks from a video and save overlay video.  Returns a DataFrame and overlay path."""
        cap = cv2.VideoCapture(video_path)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        resize = self.config.get('frame_resize', None)
        out_w = resize[0] if resize else width
        out_h = resize[1] if resize else height
        overlay_path = os.path.join(self.config['output_dir'], os.path.basename(video_path) + '_hand_overlay.mp4')
        writer = cv2.VideoWriter(overlay_path, cv2.VideoWriter_fourcc(*'mp4v'), fps or 30.0, (out_w, out_h))
        finger_keys = ['thumb_tip','index_tip','middle_tip','ring_tip','pinky_tip']
        deque_buffer: Dict[str, List[Tuple[float,float]]] = {k: [] for k in finger_keys}
        rows: List[Dict] = []
        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if resize:
                frame_resized = cv2.resize(frame, (out_w, out_h))
            else:
                frame_resized = frame
            rgb_frame = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            try:
                result = self.landmarker.detect(mp_image)
            except Exception:
                result = None
            row: Dict = {'frame': frame_idx}
            for fk in finger_keys:
                row[fk + '_x'] = None
                row[fk + '_y'] = None
                row[fk + '_conf'] = 0.0
            if result and result.hand_landmarks:
                hand_lm = result.hand_landmarks[0]
                handedness = result.handedness[0].category_name if result.handedness else 'Unknown'
                lm_idx_map = {'thumb_tip':4, 'index_tip':8, 'middle_tip':12, 'ring_tip':16, 'pinky_tip':20}
                for fk, idx in lm_idx_map.items():
                    lm = hand_lm[idx]
                    x_px = lm.x * out_w
                    y_px = lm.y * out_h
                    confidence = getattr(hand_lm[idx], 'presence', 1.0)
                    row[fk + '_x'] = x_px
                    row[fk + '_y'] = y_px
                    row[fk + '_conf'] = confidence
                    buf = deque_buffer[fk]
                    buf.append((x_px, y_px))
                    if len(buf) > self.smoothing_window:
                        buf.pop(0)
                    xs = [pt[0] for pt in buf]
                    ys = [pt[1] for pt in buf]
                    row[fk + '_x_smooth'] = float(np.mean(xs))
                    row[fk + '_y_smooth'] = float(np.mean(ys))
                row['handedness'] = handedness
            else:
                for fk in finger_keys:
                    buf = deque_buffer[fk]
                    buf.append((None, None))
                    if len(buf) > self.smoothing_window:
                        buf.pop(0)
                    xs = [pt[0] for pt in buf if pt[0] is not None]
                    ys = [pt[1] for pt in buf if pt[1] is not None]
                    row[fk + '_x_smooth'] = float(np.mean(xs)) if xs else None
                    row[fk + '_y_smooth'] = float(np.mean(ys)) if ys else None
                row['handedness'] = 'Unknown'
            rows.append(row)
            # Draw overlay
            overlay_frame = frame_resized.copy()
            for fk in finger_keys:
                x_s = row.get(fk + '_x_smooth', None)
                y_s = row.get(fk + '_y_smooth', None)
                if x_s is not None and y_s is not None:
                    color = (0, 255, 0)
                    cv2.circle(overlay_frame, (int(x_s), int(y_s)), 5, color, -1)
                    cv2.putText(overlay_frame, fk.split('_')[0], (int(x_s)+5, int(y_s)-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            writer.write(cv2.cvtColor(overlay_frame, cv2.COLOR_RGB2BGR))
            frame_idx += 1
            if quick_test and frame_idx >= self.config.get('quick_test_frames', 150):
                break
        cap.release()
        writer.release()
        df = pd.DataFrame(rows)
        csv_path = os.path.join(self.config['output_dir'], os.path.basename(video_path) + '_hand_landmarks.csv')
        df.to_csv(csv_path, index=False)
        json_path = os.path.join(self.config['output_dir'], os.path.basename(video_path) + '_hand_landmarks.json')
        df.to_json(json_path, orient='records', indent=2)
        print(f"Hand landmark extraction complete. CSV saved to {csv_path}")
        return df, overlay_path

# Run hand landmark extraction on uploaded videos
hand_pipelines = {}
hand_results = {}
for name, path in VIDEO_PATHS.items():
    pipeline = HandLandmarkPipeline(CONFIG)
    hand_pipelines[name] = pipeline
    df, overlay = pipeline.process_video(path, quick_test=True)
    hand_results[name] = {'df': df, 'overlay_path': overlay}


# Section 3: Ball Detection and Tracking

Detect and track the baseball using Ultralytics YOLO.  If YOLO misses the ball, a fallback based on OpenCV's Hough circle transform attempts to locate the ball. The result is a DataFrame recording ball centre, radius, and confidence per frame, along with a QA overlay video showing the tracked trajectory.

In [ ]:

import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from typing import Tuple, Dict, List

class BallTracker:
    """Detect and track a baseball using YOLO with a Hough circle fallback."""
    def __init__(self, config: Dict):
        self.config = config
        self.model = YOLO('yolov8n.pt')
        self.class_id = 32  # sports ball in COCO
        self.conf_thres = config.get('ball_conf_thresh', 0.25)
        self.frame_resize = config.get('frame_resize', None)
        self.output_dir = config.get('output_dir', '.')

    def detect_hough(self, gray: np.ndarray):
        circles = cv2.HoughCircles(
            gray,
            cv2.HOUGH_GRADIENT,
            dp=1.2,
            minDist=20,
            param1=50,
            param2=30,
            minRadius=5,
            maxRadius=50
        )
        if circles is not None:
            circles = np.round(circles[0, :]).astype('int')
            return circles[0]
        return None

    def process_video(self, video_path: str, quick_test: bool = False) -> Tuple[pd.DataFrame, str]:
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        resize = self.frame_resize
        out_w, out_h = (resize if resize else (width, height))
        overlay_path = os.path.join(self.output_dir, os.path.basename(video_path) + '_ball_overlay.mp4')
        writer = cv2.VideoWriter(overlay_path, cv2.VideoWriter_fourcc(*'mp4v'), fps or 30.0, (out_w, out_h))
        rows: List[Dict] = []
        trail_points: List[Tuple[int,int]] = []
        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_resized = cv2.resize(frame, (out_w, out_h)) if resize else frame.copy()
            results = self.model(frame_resized, verbose=False)
            detections = results[0].boxes if results and len(results) > 0 else None
            ball_x = ball_y = ball_r = None
            ball_conf = 0.0
            if detections is not None and len(detections) > 0:
                for det in detections:
                    if int(det.cls[0]) == self.class_id and float(det.conf[0]) >= self.conf_thres:
                        x1, y1, x2, y2 = det.xyxy[0].cpu().numpy()
                        ball_x = (x1 + x2) / 2
                        ball_y = (y1 + y2) / 2
                        ball_r = max((x2 - x1), (y2 - y1)) / 2
                        ball_conf = float(det.conf[0])
                        break
            if ball_x is None:
                gray = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2GRAY)
                gray = cv2.medianBlur(gray, 5)
                circle = self.detect_hough(gray)
                if circle is not None:
                    ball_x, ball_y, ball_r = circle
                    ball_conf = 0.5
            rows.append({
                'frame': frame_idx,
                'ball_x': float(ball_x) if ball_x is not None else None,
                'ball_y': float(ball_y) if ball_y is not None else None,
                'ball_r': float(ball_r) if ball_r is not None else None,
                'ball_conf': ball_conf
            })
            overlay = frame_resized.copy()
            if ball_x is not None and ball_y is not None and ball_r is not None:
                center = (int(ball_x), int(ball_y))
                radius = int(ball_r)
                cv2.circle(overlay, center, radius, (0, 0, 255), 2)
                trail_points.append(center)
            for i in range(1, len(trail_points)):
                if trail_points[i-1] and trail_points[i]:
                    cv2.line(overlay, trail_points[i-1], trail_points[i], (255, 0, 0), 2)
            writer.write(overlay)
            frame_idx += 1
            if quick_test and frame_idx >= self.config.get('quick_test_frames', 150):
                break
        cap.release()
        writer.release()
        df = pd.DataFrame(rows)
        csv_path = os.path.join(self.output_dir, os.path.basename(video_path) + '_ball_tracking.csv')
        df.to_csv(csv_path, index=False)
        json_path = os.path.join(self.output_dir, os.path.basename(video_path) + '_ball_tracking.json')
        df.to_json(json_path, orient='records', indent=2)
        print(f"Ball tracking complete. CSV saved to {csv_path}")
        return df, overlay_path

# Run ball tracking on uploaded videos
ball_trackers = {}
ball_results = {}
for name, path in VIDEO_PATHS.items():
    tracker = BallTracker(CONFIG)
    ball_trackers[name] = tracker
    df, overlay = tracker.process_video(path, quick_test=True)
    ball_results[name] = {'df': df, 'overlay_path': overlay}


# Section 4: Release Window Detection

Estimate when the ball leaves the hand by analysing both the distance between the index finger and ball centre and the ball's acceleration.  Frames with the highest combined score are selected as candidate release moments.  A montage of these frames is saved for visual inspection.

In [ ]:

import numpy as np
import cv2
import math
from typing import List, Tuple, Dict

def detect_release_windows(landmarks_df: pd.DataFrame, ball_df: pd.DataFrame, fps: float, top_n: int = 3) -> Tuple[List[int], List[float]]:
    """Return frames with the highest likelihood of ball release based on distance and acceleration."""
    df = pd.merge(landmarks_df, ball_df, on='frame', how='inner')
    df['dist'] = np.sqrt(
        (df['index_tip_x_smooth'] - df['ball_x'])**2 +
        (df['index_tip_y_smooth'] - df['ball_y'])**2
    )
    df['vel'] = np.sqrt((df['ball_x'].diff())**2 + (df['ball_y'].diff())**2) * fps
    df['acc'] = df['vel'].diff() * fps
    dist_norm = (df['dist'] - df['dist'].min()) / (df['dist'].max() - df['dist'].min() + 1e-6)
    acc_norm = (df['acc'] - df['acc'].min()) / (df['acc'].max() - df['acc'].min() + 1e-6)
    combined = (1 - dist_norm) + acc_norm
    df['release_score'] = combined
    candidates = df.nlargest(top_n, 'release_score')[['frame', 'release_score']]
    return candidates['frame'].astype(int).tolist(), candidates['release_score'].tolist()

def save_release_montage(video_path: str, candidate_frames: List[int], out_dir: str, resize: Tuple[int,int] = None) -> str:
    """Save a montage image of the candidate release frames."""
    cap = cv2.VideoCapture(video_path)
    frames: List[np.ndarray] = []
    for f in candidate_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, f)
        ret, frame = cap.read()
        if not ret:
            continue
        if resize:
            frame = cv2.resize(frame, resize)
        frames.append(frame)
    cap.release()
    if not frames:
        return ''
    montage = np.concatenate(frames, axis=1)
    montage_path = os.path.join(out_dir, os.path.basename(video_path) + '_release_montage.jpg')
    cv2.imwrite(montage_path, montage)
    return montage_path

# Detect release frames and save montages for each video
release_info: Dict[str, Dict] = {}
for name in VIDEO_PATHS:
    hand_df = hand_results[name]['df']
    ball_df = ball_results[name]['df']
    fps = VIDEO_METADATA[name]['fps']
    frames, scores = detect_release_windows(hand_df, ball_df, fps, top_n=3)
    montage = save_release_montage(VIDEO_PATHS[name], frames, CONFIG['output_dir'], CONFIG.get('frame_resize', None))
    release_info[name] = {'frames': frames, 'scores': scores, 'montage': montage}
    print(f"Candidate release frames for {name}: {frames} with scores {scores}")
    if montage:
        from IPython.display import Image as DispImage
        display(DispImage(filename=montage))


# Section 5: Seam Cue Estimation (Experimental)

For each candidate release frame, crop a region of interest (ROI) around the ball, enhance it, and compute a seam confidence score using simple edge heuristics.  Frames with seam cues above a threshold are flagged as usable; otherwise the pipeline continues gracefully.

In [ ]:

import cv2
import numpy as np
import pandas as pd
from typing import List, Tuple

def enhance_roi(roi: np.ndarray) -> np.ndarray:
    """Enhance a cropped ball ROI using contrast and sharpening."""
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    eq = clahe.apply(gray)
    blur = cv2.GaussianBlur(eq, (0,0), sigmaX=2)
    sharpened = cv2.addWeighted(eq, 1.5, blur, -0.5, 0)
    return sharpened

def seam_score(roi_enh: np.ndarray) -> float:
    """Compute a simple seam score as the ratio of edge pixels to area."""
    edges = cv2.Canny(roi_enh, 100, 200)
    edge_sum = np.sum(edges > 0)
    area = roi_enh.size
    return edge_sum / area if area > 0 else 0.0

def estimate_seam_cues(video_path: str, ball_df: pd.DataFrame, release_frames: List[int], radius_scale: float = 1.5) -> Tuple[List[float], List[bool]]:
    """Compute seam confidence scores and usability flags for frames around release."""
    cap = cv2.VideoCapture(video_path)
    scores: List[float] = []
    usable: List[bool] = []
    for f in release_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, f)
        ret, frame = cap.read()
        if not ret:
            scores.append(0.0)
            usable.append(False)
            continue
        row = ball_df[ball_df['frame'] == f]
        if row.empty or pd.isna(row.iloc[0]['ball_x']):
            scores.append(0.0)
            usable.append(False)
            continue
        x = int(row.iloc[0]['ball_x'])
        y = int(row.iloc[0]['ball_y'])
        r = int(row.iloc[0]['ball_r'] * radius_scale)
        h, w, _ = frame.shape
        x1 = max(x - r, 0)
        y1 = max(y - r, 0)
        x2 = min(x + r, w - 1)
        y2 = min(y + r, h - 1)
        roi = frame[y1:y2, x1:x2]
        roi_enh = enhance_roi(roi)
        score = seam_score(roi_enh)
        threshold = 0.02  # heuristic threshold
        scores.append(score)
        usable.append(score > threshold)
    cap.release()
    return scores, usable

# Compute seam cues
seam_info: Dict[str, Dict] = {}
for name in VIDEO_PATHS:
    frames = release_info[name]['frames']
    if not frames:
        seam_info[name] = {'scores': [], 'usable': []}
        continue
    scores, usable = estimate_seam_cues(VIDEO_PATHS[name], ball_results[name]['df'], frames)
    seam_info[name] = {'scores': scores, 'usable': usable}
    for f, s, u in zip(frames, scores, usable):
        print(f"Video {name}, frame {f}: seam score {s:.4f}, usable: {u}")


# Section 6: Ball and Finger Geometry Reconstruction

Project fingertip coordinates onto a 3D sphere approximating the baseball.  Compute geodesic distances between finger contacts, thumb opposition angles, and a simple pressure proxy based on contact duration and acceleration.  If camera calibration is available, more accurate mapping may be substituted.

In [ ]:

import numpy as np
import pandas as pd
import math
from typing import Dict, List

def project_fingers_to_sphere(hand_df: pd.DataFrame, ball_df: pd.DataFrame, frames: List[int]) -> Dict[int, Dict[str, np.ndarray]]:
    """Map 2D fingertip coordinates onto a unit sphere centred at the ball centre."""
    result: Dict[int, Dict[str, np.ndarray]] = {}
    for f in frames:
        row_hand = hand_df[hand_df['frame'] == f]
        row_ball = ball_df[ball_df['frame'] == f]
        if row_hand.empty or row_ball.empty:
            continue
        bx = row_ball.iloc[0]['ball_x']
        by = row_ball.iloc[0]['ball_y']
        br = row_ball.iloc[0]['ball_r']
        if pd.isna(bx) or pd.isna(br):
            continue
        sphere_points: Dict[str, np.ndarray] = {}
        for fk in ['thumb_tip','index_tip','middle_tip','ring_tip','pinky_tip']:
            x = row_hand.iloc[0][fk + '_x_smooth']
            y = row_hand.iloc[0][fk + '_y_smooth']
            if pd.isna(x) or pd.isna(y):
                sphere_points[fk] = None
                continue
            dx = (x - bx) / br
            dy = (y - by) / br
            dist_sq = dx*dx + dy*dy
            if dist_sq > 1.0:
                factor = 1.0 / math.sqrt(dist_sq)
                dx *= factor
                dy *= factor
                dz = 0.0
            else:
                dz = math.sqrt(1.0 - dist_sq)
            sphere_points[fk] = np.array([dx, dy, dz])
        result[f] = sphere_points
    return result

def compute_geodesic_features(sphere_points: Dict[str, np.ndarray], radius: float = 1.0) -> Dict[str, float]:
    """Compute geodesic distances and thumb opposition from mapped points."""
    features: Dict[str, float] = {}
    pairs = [('thumb_tip','index_tip'), ('index_tip','middle_tip'), ('middle_tip','ring_tip'), ('ring_tip','pinky_tip'), ('thumb_tip','pinky_tip')]
    for a, b in pairs:
        p1 = sphere_points.get(a)
        p2 = sphere_points.get(b)
        key = f"{a}_to_{b}_dist"
        if p1 is None or p2 is None:
            features[key] = None
        else:
            cos_angle = np.clip(np.dot(p1, p2) / (radius*radius), -1.0, 1.0)
            angle = math.acos(cos_angle)
            features[key] = angle * radius
    thumb = sphere_points.get('thumb_tip')
    index = sphere_points.get('index_tip')
    middle = sphere_points.get('middle_tip')
    if thumb is not None and index is not None and middle is not None:
        v1 = index - thumb
        v2 = middle - thumb
        cos_ang = np.dot(v1, v2) / ((np.linalg.norm(v1) * np.linalg.norm(v2)) + 1e-6)
        opposition_angle = math.degrees(math.acos(np.clip(cos_ang, -1.0, 1.0)))
        features['thumb_opposition_deg'] = opposition_angle
    else:
        features['thumb_opposition_deg'] = None
    return features

def compute_pressure_proxy(hand_df: pd.DataFrame, ball_df: pd.DataFrame, release_frames: List[int], fps: float) -> float:
    """Estimate a grip pressure proxy from contact duration and acceleration."""
    contact_frames = 0
    for i in range(len(hand_df)):
        row_hand = hand_df.iloc[i]
        row_ball = ball_df.iloc[i] if i < len(ball_df) else None
        if row_ball is None or pd.isna(row_ball['ball_x']):
            continue
        dist = math.hypot(
            row_hand['index_tip_x_smooth'] - row_ball['ball_x'],
            row_hand['index_tip_y_smooth'] - row_ball['ball_y']
        )
        if dist < 2 * row_ball['ball_r']:
            contact_frames += 1
        if row_hand['frame'] in release_frames:
            break
    contact_duration = contact_frames / fps if fps > 0 else 0.0
    df_merged = pd.merge(hand_df, ball_df, on='frame', how='inner')
    if 'vel' in df_merged.columns and not df_merged.empty:
        acc = df_merged['vel'].iloc[release_frames[0]] if release_frames else 0.0
    else:
        acc = 0.0
    return contact_duration * acc

# Compute geometry features and pressure proxies for each video
geometry_info: Dict[str, List[Dict]] = {}
for name in VIDEO_PATHS:
    frames = release_info[name]['frames']
    hand_df = hand_results[name]['df']
    ball_df = ball_results[name]['df']
    sp_per_frame = project_fingers_to_sphere(hand_df, ball_df, frames)
    feats: List[Dict] = []
    for f in frames:
        sp = sp_per_frame.get(f, None)
        if sp is None:
            continue
        g_feats = compute_geodesic_features(sp)
        g_feats['frame'] = f
        feats.append(g_feats)
    geometry_info[name] = feats
    proxy_score = compute_pressure_proxy(hand_df, ball_df, frames, VIDEO_METADATA[name]['fps'])
    geometry_info[name + '_pressure_proxy'] = proxy_score
    print(f"Geometry features for {name}: {feats}")
    print(f"Pressure proxy score for {name}: {proxy_score}")


# Section 7: Interactive 3D Visualization

Visualise the reconstructed baseball and fingertip contact points in 3D using Plotly.  The sphere represents the ball, and coloured markers denote the positions of each fingertip at the candidate release frames.  Interactive rotation and zoom enable detailed inspection.

In [ ]:

import plotly.graph_objects as go
import numpy as np
from typing import Dict, List

def create_sphere_mesh(resolution: int = 30):
    """Generate x, y, z coordinates for a unit sphere mesh."""
    theta = np.linspace(0, 2 * np.pi, resolution)
    phi = np.linspace(0, np.pi, resolution)
    theta, phi = np.meshgrid(theta, phi)
    x = np.sin(phi) * np.cos(theta)
    y = np.sin(phi) * np.sin(theta)
    z = np.cos(phi)
    return x, y, z

def visualize_grip(sphere_points: Dict[int, Dict[str, np.ndarray]], frames: List[int]) -> go.Figure:
    """Render the baseball and fingertip points in 3D using Plotly."""
    x, y, z = create_sphere_mesh()
    fig = go.Figure()
    fig.add_trace(go.Surface(x=x, y=y, z=z, opacity=0.3, colorscale='Viridis', showscale=False))
    finger_colors = {
        'thumb_tip': 'red',
        'index_tip': 'blue',
        'middle_tip': 'green',
        'ring_tip': 'orange',
        'pinky_tip': 'purple'
    }
    for fk, color in finger_colors.items():
        xs = [sphere_points[f][fk][0] if sphere_points[f].get(fk) is not None else None for f in frames]
        ys = [sphere_points[f][fk][1] if sphere_points[f].get(fk) is not None else None for f in frames]
        zs = [sphere_points[f][fk][2] if sphere_points[f].get(fk) is not None else None for f in frames]
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode='markers+lines',
            marker=dict(size=5, color=color),
            name=fk
        ))
    fig.update_layout(
        scene=dict(aspectmode='data'),
        title="3D Baseball Grip Visualization",
        legend=dict(x=0.0, y=1.0)
    )
    return fig

# Display 3D visualisation for the first video with release frames
for name in VIDEO_PATHS:
    frames = release_info[name]['frames']
    if frames:
        sp = project_fingers_to_sphere(hand_results[name]['df'], ball_results[name]['df'], frames)
        fig = visualize_grip(sp, frames)
        fig.show()
        break


# Section 8: Training Data Export

Aggregate the features extracted throughout the pipeline into a single training table. Each row corresponds to a pitch attempt and includes metadata, release metrics, grip geometry features, seam confidence, and a pressure proxy.  The dataset is exported in both CSV and JSONL formats for downstream model training.  Coverage statistics summarise the availability of usable seam frames.

In [ ]:

import json

def create_training_dataset(video_names: List[str]) -> pd.DataFrame:
    """Combine extracted metrics into a flat training dataset."""
    rows: List[Dict] = []
    for name in video_names:
        hand_df = hand_results[name]['df']
        ball_df = ball_results[name]['df']
        release_frames = release_info[name]['frames']
        seam_scores = seam_info[name]['scores']
        seam_usable = seam_info[name]['usable']
        geometry_feats = geometry_info[name]
        proxy_score = geometry_info.get(name + '_pressure_proxy', 0.0)
        for i, f in enumerate(release_frames):
            geom = next((g for g in geometry_feats if g['frame'] == f), {})
            row: Dict = {
                'video_id': name,
                'pitcher_id': None,
                'handedness': hand_df.iloc[0].get('handedness', 'Unknown'),
                'pitch_type': None,
                'release_frame': f,
                'ball_center_x': ball_df.loc[ball_df['frame'] == f, 'ball_x'].values[0] if not ball_df.loc[ball_df['frame'] == f].empty else None,
                'ball_center_y': ball_df.loc[ball_df['frame'] == f, 'ball_y'].values[0] if not ball_df.loc[ball_df['frame'] == f].empty else None,
                'ball_radius_px': ball_df.loc[ball_df['frame'] == f, 'ball_r'].values[0] if not ball_df.loc[ball_df['frame'] == f].empty else None,
                'seam_score': seam_scores[i] if seam_scores else None,
                'seam_usable': seam_usable[i] if seam_usable else False,
                'pressure_proxy': proxy_score
            }
            for key, val in geom.items():
                if key != 'frame':
                    row[key] = val
            rows.append(row)
    return pd.DataFrame(rows)

# Build and save training dataset
training_df = create_training_dataset(list(VIDEO_PATHS.keys()))
train_csv = os.path.join(CONFIG['output_dir'], 'training_data.csv')
training_df.to_csv(train_csv, index=False)
train_jsonl = os.path.join(CONFIG['output_dir'], 'training_data.jsonl')
with open(train_jsonl, 'w') as f:
    for _, row in training_df.iterrows():
        f.write(json.dumps(row.dropna().to_dict()) + "
")
print(f"Training data saved to {train_csv} and {train_jsonl}")
print(training_df[['video_id','seam_usable']].groupby('video_id').agg(['count','sum']))


# Section 9: Evaluation & Error Analysis

Summarise pipeline performance by computing detection rates for hands and the ball, missing frame counts, and seam cue yields.  Generate a gallery of low‑confidence frames to diagnose failure modes and suggest improvements.

In [ ]:

import matplotlib.pyplot as plt
from IPython.display import Image as DispImage, display

def evaluate_results(hand_df: pd.DataFrame, ball_df: pd.DataFrame, seam_scores: List[float], seam_usable: List[bool]) -> Dict[str, float]:
    """Compute detection rates, missing frame counts, and seam yield."""
    total_frames = len(hand_df)
    hand_detected = hand_df[[col for col in hand_df.columns if col.endswith('_x')]].notna().any(axis=1).sum()
    ball_detected = ball_df['ball_x'].notna().sum()
    detection_rate_hand = hand_detected / total_frames if total_frames else 0.0
    detection_rate_ball = ball_detected / total_frames if total_frames else 0.0
    missing_frames = total_frames - min(len(hand_df), len(ball_df))
    seam_yield = sum(seam_usable) / len(seam_usable) if seam_usable else 0.0
    return {
        'hand_detection_rate': detection_rate_hand,
        'ball_detection_rate': detection_rate_ball,
        'missing_frames': missing_frames,
        'seam_yield': seam_yield
    }

def create_failure_gallery(video_path: str, hand_df: pd.DataFrame, ball_df: pd.DataFrame, threshold: float = 0.25, max_imgs: int = 5) -> List[str]:
    """Save frames where either hand or ball detection confidence is low."""
    cap = cv2.VideoCapture(video_path)
    failure_paths: List[str] = []
    count = 0
    for i in range(min(len(hand_df), len(ball_df))):
        if count >= max_imgs:
            break
        row_h = hand_df.iloc[i]
        row_b = ball_df.iloc[i]
        # Determine low confidence for hand and ball
        hand_confs = [row_h[col] for col in row_h.index if col.endswith('_conf')]
        max_hand_conf = max(hand_confs) if hand_confs else 0.0
        low_hand = max_hand_conf < threshold
        low_ball = (row_b['ball_conf'] < threshold) if not pd.isna(row_b['ball_conf']) else True
        if low_hand or low_ball:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                path = os.path.join(CONFIG['output_dir'], f"failure_{os.path.basename(video_path)}_{i:04d}.jpg")
                cv2.imwrite(path, frame)
                failure_paths.append(path)
                count += 1
    cap.release()
    return failure_paths

# Evaluate and create failure galleries for each video
evaluation_metrics: Dict[str, Dict] = {}
failure_galleries: Dict[str, List[str]] = {}
for name in VIDEO_PATHS:
    metrics = evaluate_results(hand_results[name]['df'], ball_results[name]['df'], seam_info[name]['scores'], seam_info[name]['usable'])
    evaluation_metrics[name] = metrics
    print(f"Evaluation metrics for {name}: {metrics}")
    failures = create_failure_gallery(VIDEO_PATHS[name], hand_results[name]['df'], ball_results[name]['df'], threshold=0.3, max_imgs=5)
    failure_galleries[name] = failures
    print(f"Failure frames saved for {name}: {failures}")


# Section 10: Migration Kit for VS Code

Write the core logic into separate Python modules for easier migration to a production environment.  A requirements file and README are generated to guide installation and provide TODO steps for converting the notebook into a web service with FastAPI and a React front‑end.

In [ ]:

import os

# Directory to store exported modules and documentation
module_dir = os.path.join(CONFIG['output_dir'], 'modules')
os.makedirs(module_dir, exist_ok=True)

# Module strings
packages = [
    "mediapipe",
    "ultralytics",
    "opencv-python-headless",
    "numpy",
    "pandas",
    "plotly",
    "scipy",
    "pillow",
    "pytesseract",
    "scikit-image",
    "tqdm"
]

io_utils_py = """
import os
import cv2
import numpy as np
from typing import Dict, List, Tuple
from google.colab import files
from PIL import Image
import pytesseract

def upload_videos() -> Dict[str, str]:
    uploaded = files.upload()
    video_paths: Dict[str, str] = {}
    for name, data in uploaded.items():
        path = os.path.join('/content', name)
        with open(path, 'wb') as f:
            f.write(data)
        video_paths[name] = path
    return video_paths

def extract_metadata(video_path: str) -> Dict[str, float]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f'Cannot open video {video_path}')
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    duration = frame_count / fps if fps > 0 else 0
    cap.release()
    return {
        'fps': fps,
        'width': width,
        'height': height,
        'duration_sec': duration,
        'frame_count': int(frame_count)
    }

def sample_frames(video_path: str, stride: int = 30, resize: Tuple[int, int] = None, max_frames: int = 150) -> List[np.ndarray]:
    frames: List[np.ndarray] = []
    cap = cv2.VideoCapture(video_path)
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % stride == 0:
            if resize:
                frame = cv2.resize(frame, resize)
            frames.append(frame)
            if len(frames) >= max_frames:
                break
        idx += 1
    cap.release()
    return frames

def save_sample_frames(frames: List[np.ndarray], prefix: str, out_dir: str) -> None:
    qa_dir = os.path.join(out_dir, f"{prefix}_qa_frames")
    os.makedirs(qa_dir, exist_ok=True)
    for i, frame in enumerate(frames):
        cv2.imwrite(os.path.join(qa_dir, f'frame_{i:04d}.jpg'), frame)

def parse_ocr_overlay(frame: np.ndarray, ocr_enabled: bool = False) -> Dict[str, str]:
    if not ocr_enabled:
        return {}
    text = pytesseract.image_to_string(Image.fromarray(frame))
    return {'overlay_text': text.strip()}
"""

hand_pipeline_py = """
import os
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions
from typing import Dict, Tuple, List

class HandLandmarkPipeline:
    """Detect and smooth hand landmarks using MediaPipe."""
    def __init__(self, config: Dict):
        self.config = config
        self.model_path = config.get('hand_model_path', 'hand_landmarker.task')
        if not os.path.exists(self.model_path):
            import urllib.request
            url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker_lite/float16/1/hand_landmarker_lite.task'
            urllib.request.urlretrieve(url, self.model_path)
        options = HandLandmarkerOptions(
            base_options=vision.BaseOptions(model_asset_path=self.model_path),
            num_hands=1,
            min_hand_detection_confidence=config.get('hand_conf_thresh', 0.5),
            min_hand_presence_confidence=config.get('hand_conf_thresh', 0.5),
            min_tracking_confidence=config.get('hand_conf_thresh', 0.5)
        )
        self.landmarker = HandLandmarker.create_from_options(options)
        self.smoothing_window = config.get('smoothing_window', 7)
        self.frame_resize = config.get('frame_resize', None)
        self.output_dir = config.get('output_dir', '.')

    def process_video(self, video_path: str, quick_test: bool = False) -> Tuple[pd.DataFrame, str]:
        cap = cv2.VideoCapture(video_path)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        resize = self.frame_resize
        out_w, out_h = (resize if resize else (width, height))
        overlay_path = os.path.join(self.output_dir, os.path.basename(video_path) + '_hand_overlay.mp4')
        writer = cv2.VideoWriter(overlay_path, cv2.VideoWriter_fourcc(*'mp4v'), fps or 30.0, (out_w, out_h))
        finger_keys = ['thumb_tip','index_tip','middle_tip','ring_tip','pinky_tip']
        deque_buffer: Dict[str, List[Tuple[float,float]]] = {k: [] for k in finger_keys}
        rows: List[Dict] = []
        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_resized = cv2.resize(frame, (out_w, out_h)) if resize else frame
            rgb_frame = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            try:
                result = self.landmarker.detect(mp_image)
            except Exception:
                result = None
            row: Dict = {'frame': frame_idx}
            for fk in finger_keys:
                row[fk + '_x'] = None
                row[fk + '_y'] = None
                row[fk + '_conf'] = 0.0
            if result and result.hand_landmarks:
                hand_lm = result.hand_landmarks[0]
                handedness = result.handedness[0].category_name if result.handedness else 'Unknown'
                lm_idx_map = {'thumb_tip':4, 'index_tip':8, 'middle_tip':12, 'ring_tip':16, 'pinky_tip':20}
                for fk, idx in lm_idx_map.items():
                    lm = hand_lm[idx]
                    x_px = lm.x * out_w
                    y_px = lm.y * out_h
                    confidence = getattr(hand_lm[idx], 'presence', 1.0)
                    row[fk + '_x'] = x_px
                    row[fk + '_y'] = y_px
                    row[fk + '_conf'] = confidence
                    buf = deque_buffer[fk]
                    buf.append((x_px, y_px))
                    if len(buf) > self.smoothing_window:
                        buf.pop(0)
                    xs = [pt[0] for pt in buf]
                    ys = [pt[1] for pt in buf]
                    row[fk + '_x_smooth'] = float(np.mean(xs))
                    row[fk + '_y_smooth'] = float(np.mean(ys))
                row['handedness'] = handedness
            else:
                for fk in finger_keys:
                    buf = deque_buffer[fk]
                    buf.append((None, None))
                    if len(buf) > self.smoothing_window:
                        buf.pop(0)
                    xs = [pt[0] for pt in buf if pt[0] is not None]
                    ys = [pt[1] for pt in buf if pt[1] is not None]
                    row[fk + '_x_smooth'] = float(np.mean(xs)) if xs else None
                    row[fk + '_y_smooth'] = float(np.mean(ys)) if ys else None
                row['handedness'] = 'Unknown'
            rows.append(row)
            overlay_frame = frame_resized.copy()
            for fk in finger_keys:
                x_s = row.get(fk + '_x_smooth', None)
                y_s = row.get(fk + '_y_smooth', None)
                if x_s is not None and y_s is not None:
                    color = (0, 255, 0)
                    cv2.circle(overlay_frame, (int(x_s), int(y_s)), 5, color, -1)
                    cv2.putText(overlay_frame, fk.split('_')[0], (int(x_s)+5, int(y_s)-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            writer.write(cv2.cvtColor(overlay_frame, cv2.COLOR_RGB2BGR))
            frame_idx += 1
            if quick_test and frame_idx >= self.config.get('quick_test_frames', 150):
                break
        cap.release()
        writer.release()
        df = pd.DataFrame(rows)
        csv_path = os.path.join(self.output_dir, os.path.basename(video_path) + '_hand_landmarks.csv')
        df.to_csv(csv_path, index=False)
        json_path = os.path.join(self.output_dir, os.path.basename(video_path) + '_hand_landmarks.json')
        df.to_json(json_path, orient='records', indent=2)
        return df, overlay_path
"""

ball_pipeline_py = """
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from typing import Dict, Tuple, List

class BallTracker:
    """Detect and track a baseball using YOLO with a Hough circle fallback."""
    def __init__(self, config: Dict):
        self.config = config
        self.model = YOLO('yolov8n.pt')
        self.class_id = 32
        self.conf_thres = config.get('ball_conf_thresh', 0.25)
        self.frame_resize = config.get('frame_resize', None)
        self.output_dir = config.get('output_dir', '.')

    def detect_hough(self, gray: np.ndarray):
        circles = cv2.HoughCircles(
            gray,
            cv2.HOUGH_GRADIENT,
            dp=1.2,
            minDist=20,
            param1=50,
            param2=30,
            minRadius=5,
            maxRadius=50
        )
        if circles is not None:
            circles = np.round(circles[0, :]).astype('int')
            return circles[0]
        return None

    def process_video(self, video_path: str, quick_test: bool = False) -> Tuple[pd.DataFrame, str]:
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        resize = self.frame_resize
        out_w, out_h = (resize if resize else (width, height))
        overlay_path = os.path.join(self.output_dir, os.path.basename(video_path) + '_ball_overlay.mp4')
        writer = cv2.VideoWriter(overlay_path, cv2.VideoWriter_fourcc(*'mp4v'), fps or 30.0, (out_w, out_h))
        rows: List[Dict] = []
        trail_points: List[Tuple[int,int]] = []
        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_resized = cv2.resize(frame, (out_w, out_h)) if resize else frame.copy()
            results = self.model(frame_resized, verbose=False)
            detections = results[0].boxes if results and len(results) > 0 else None
            ball_x = ball_y = ball_r = None
            ball_conf = 0.0
            if detections is not None and len(detections) > 0:
                for det in detections:
                    if int(det.cls[0]) == self.class_id and float(det.conf[0]) >= self.conf_thres:
                        x1, y1, x2, y2 = det.xyxy[0].cpu().numpy()
                        ball_x = (x1 + x2) / 2
                        ball_y = (y1 + y2) / 2
                        ball_r = max((x2 - x1), (y2 - y1)) / 2
                        ball_conf = float(det.conf[0])
                        break
            if ball_x is None:
                gray = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2GRAY)
                gray = cv2.medianBlur(gray, 5)
                circle = self.detect_hough(gray)
                if circle is not None:
                    ball_x, ball_y, ball_r = circle
                    ball_conf = 0.5
            rows.append({
                'frame': frame_idx,
                'ball_x': float(ball_x) if ball_x is not None else None,
                'ball_y': float(ball_y) if ball_y is not None else None,
                'ball_r': float(ball_r) if ball_r is not None else None,
                'ball_conf': ball_conf
            })
            overlay = frame_resized.copy()
            if ball_x is not None and ball_y is not None and ball_r is not None:
                center = (int(ball_x), int(ball_y))
                radius = int(ball_r)
                cv2.circle(overlay, center, radius, (0, 0, 255), 2)
                trail_points.append(center)
            for i in range(1, len(trail_points)):
                if trail_points[i-1] and trail_points[i]:
                    cv2.line(overlay, trail_points[i-1], trail_points[i], (255, 0, 0), 2)
            writer.write(overlay)
            frame_idx += 1
            if quick_test and frame_idx >= self.config.get('quick_test_frames', 150):
                break
        cap.release()
        writer.release()
        df = pd.DataFrame(rows)
        csv_path = os.path.join(self.output_dir, os.path.basename(video_path) + '_ball_tracking.csv')
        df.to_csv(csv_path, index=False)
        json_path = os.path.join(self.output_dir, os.path.basename(video_path) + '_ball_tracking.json')
        df.to_json(json_path, orient='records', indent=2)
        return df, overlay_path
"""

seam_pipeline_py = """
import cv2
import numpy as np
import pandas as pd
from typing import List, Tuple

def enhance_roi(roi: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    eq = clahe.apply(gray)
    blur = cv2.GaussianBlur(eq, (0,0), sigmaX=2)
    sharpened = cv2.addWeighted(eq, 1.5, blur, -0.5, 0)
    return sharpened

def seam_score(roi_enh: np.ndarray) -> float:
    edges = cv2.Canny(roi_enh, 100, 200)
    edge_sum = np.sum(edges > 0)
    area = roi_enh.size
    return edge_sum / area if area > 0 else 0.0

def estimate_seam_cues(video_path: str, ball_df: pd.DataFrame, frames: List[int], radius_scale: float = 1.5, threshold: float = 0.02) -> Tuple[List[float], List[bool]]:
    cap = cv2.VideoCapture(video_path)
    scores: List[float] = []
    usable: List[bool] = []
    for f in frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, f)
        ret, frame = cap.read()
        if not ret:
            scores.append(0.0)
            usable.append(False)
            continue
        row = ball_df[ball_df['frame'] == f]
        if row.empty or pd.isna(row.iloc[0]['ball_x']):
            scores.append(0.0)
            usable.append(False)
            continue
        x = int(row.iloc[0]['ball_x'])
        y = int(row.iloc[0]['ball_y'])
        r = int(row.iloc[0]['ball_r'] * radius_scale)
        h, w, _ = frame.shape
        x1 = max(x - r, 0)
        y1 = max(y - r, 0)
        x2 = min(x + r, w - 1)
        y2 = min(y + r, h - 1)
        roi = frame[y1:y2, x1:x2]
        roi_enh = enhance_roi(roi)
        score = seam_score(roi_enh)
        scores.append(score)
        usable.append(score > threshold)
    cap.release()
    return scores, usable
"""

recon_pipeline_py = """
import numpy as np
import pandas as pd
import math
from typing import Dict, List

def project_fingers_to_sphere(hand_df: pd.DataFrame, ball_df: pd.DataFrame, frames: List[int]) -> Dict[int, Dict[str, np.ndarray]]:
    result: Dict[int, Dict[str, np.ndarray]] = {}
    for f in frames:
        row_hand = hand_df[hand_df['frame'] == f]
        row_ball = ball_df[ball_df['frame'] == f]
        if row_hand.empty or row_ball.empty:
            continue
        bx = row_ball.iloc[0]['ball_x']
        by = row_ball.iloc[0]['ball_y']
        br = row_ball.iloc[0]['ball_r']
        if pd.isna(bx) or pd.isna(br):
            continue
        sphere_points: Dict[str, np.ndarray] = {}
        for fk in ['thumb_tip','index_tip','middle_tip','ring_tip','pinky_tip']:
            x = row_hand.iloc[0][fk + '_x_smooth']
            y = row_hand.iloc[0][fk + '_y_smooth']
            if pd.isna(x) or pd.isna(y):
                sphere_points[fk] = None
                continue
            dx = (x - bx) / br
            dy = (y - by) / br
            dist_sq = dx*dx + dy*dy
            if dist_sq > 1.0:
                factor = 1.0 / math.sqrt(dist_sq)
                dx *= factor
                dy *= factor
                dz = 0.0
            else:
                dz = math.sqrt(1.0 - dist_sq)
            sphere_points[fk] = np.array([dx, dy, dz])
        result[f] = sphere_points
    return result

def compute_geodesic_features(sphere_points: Dict[str, np.ndarray], radius: float = 1.0) -> Dict[str, float]:
    features: Dict[str, float] = {}
    pairs = [('thumb_tip','index_tip'), ('index_tip','middle_tip'), ('middle_tip','ring_tip'), ('ring_tip','pinky_tip'), ('thumb_tip','pinky_tip')]
    for a, b in pairs:
        p1 = sphere_points.get(a)
        p2 = sphere_points.get(b)
        key = f"{a}_to_{b}_dist"
        if p1 is None or p2 is None:
            features[key] = None
        else:
            cos_angle = np.clip(np.dot(p1, p2) / (radius*radius), -1.0, 1.0)
            angle = math.acos(cos_angle)
            features[key] = angle * radius
    thumb = sphere_points.get('thumb_tip')
    index = sphere_points.get('index_tip')
    middle = sphere_points.get('middle_tip')
    if thumb is not None and index is not None and middle is not None:
        v1 = index - thumb
        v2 = middle - thumb
        cos_ang = np.dot(v1, v2) / ((np.linalg.norm(v1) * np.linalg.norm(v2)) + 1e-6)
        angle = math.acos(np.clip(cos_ang, -1.0, 1.0))
        features['thumb_opposition_deg'] = math.degrees(angle)
    else:
        features['thumb_opposition_deg'] = None
    return features

def compute_pressure_proxy(hand_df: pd.DataFrame, ball_df: pd.DataFrame, release_frames: List[int], fps: float) -> float:
    contact_frames = 0
    for i in range(len(hand_df)):
        row_hand = hand_df.iloc[i]
        row_ball = ball_df.iloc[i] if i < len(ball_df) else None
        if row_ball is None or pd.isna(row_ball['ball_x']):
            continue
        dist = math.hypot(
            row_hand['index_tip_x_smooth'] - row_ball['ball_x'],
            row_hand['index_tip_y_smooth'] - row_ball['ball_y']
        )
        if dist < 2 * row_ball['ball_r']:
            contact_frames += 1
        if row_hand['frame'] in release_frames:
            break
    contact_duration = contact_frames / fps if fps > 0 else 0.0
    df_merged = pd.merge(hand_df, ball_df, on='frame', how='inner')
    if 'vel' in df_merged.columns and not df_merged.empty:
        acc = df_merged['vel'].iloc[release_frames[0]] if release_frames else 0.0
    else:
        acc = 0.0
    return contact_duration * acc
"""

viz_utils_py = """
import numpy as np
import plotly.graph_objects as go
from typing import Dict, List

def create_sphere_mesh(resolution: int = 30):
    theta = np.linspace(0, 2 * np.pi, resolution)
    phi = np.linspace(0, np.pi, resolution)
    theta, phi = np.meshgrid(theta, phi)
    x = np.sin(phi) * np.cos(theta)
    y = np.sin(phi) * np.sin(theta)
    z = np.cos(phi)
    return x, y, z

def visualize_grip(sphere_points: Dict[int, Dict[str, np.ndarray]], frames: List[int]) -> go.Figure:
    x, y, z = create_sphere_mesh()
    fig = go.Figure()
    fig.add_trace(go.Surface(x=x, y=y, z=z, opacity=0.3, colorscale='Viridis', showscale=False))
    finger_colors = {
        'thumb_tip': 'red',
        'index_tip': 'blue',
        'middle_tip': 'green',
        'ring_tip': 'orange',
        'pinky_tip': 'purple'
    }
    for fk, color in finger_colors.items():
        xs = [sphere_points[f][fk][0] if sphere_points[f].get(fk) is not None else None for f in frames]
        ys = [sphere_points[f][fk][1] if sphere_points[f].get(fk) is not None else None for f in frames]
        zs = [sphere_points[f][fk][2] if sphere_points[f].get(fk) is not None else None for f in frames]
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode='markers+lines',
            marker=dict(size=5, color=color),
            name=fk
        ))
    fig.update_layout(
        scene=dict(aspectmode='data'),
        title='3D Baseball Grip Visualization',
        legend=dict(x=0.0, y=1.0)
    )
    return fig
"""

# Write out modules
modules = {
    'io_utils.py': io_utils_py,
    'hand_pipeline.py': hand_pipeline_py,
    'ball_pipeline.py': ball_pipeline_py,
    'seam_pipeline.py': seam_pipeline_py,
    'recon_pipeline.py': recon_pipeline_py,
    'viz_utils.py': viz_utils_py
}
for fname, content in modules.items():
    with open(os.path.join(module_dir, fname), 'w') as fh:
        fh.write(content)

# Write requirements.txt
with open(os.path.join(module_dir, 'requirements.txt'), 'w') as f:
    f.write("
".join(packages))

# Write README.md
readme_content = """
# Pitch Grip Pipeline Modules

This directory contains Python modules extracted from the Colab prototype for baseball pitch grip analysis. Each module corresponds to a section of the notebook and is ready to be imported into other projects.

## Modules

- `io_utils.py`: Video I/O utilities including file upload, metadata extraction and frame sampling.
- `hand_pipeline.py`: Hand landmark detection and smoothing using MediaPipe.
- `ball_pipeline.py`: Baseball detection and tracking with YOLO and a Hough circle fallback.
- `seam_pipeline.py`: Seam cue estimation via ROI enhancement and edge heuristics.
- `recon_pipeline.py`: 3D projection of fingertips onto a sphere and computation of grip features.
- `viz_utils.py`: Helper functions to visualise the ball and finger contacts in 3D using Plotly.

## Installation

Install the required packages using the provided `requirements.txt` file:

```bash
pip install -r requirements.txt
```

## Usage

Import the modules into your Python project or Jupyter notebook and use the classes/functions as demonstrated in the Colab prototype. The modules are designed to be self‑contained and modular so that they can be integrated into a backend API or other applications.

## TODO

- Refactor the notebook into a FastAPI backend service.
- Build a React front‑end to interact with the API and visualise results.
- Add unit tests for all modules and integrate continuous integration.
"""
with open(os.path.join(module_dir, 'README.md'), 'w') as f:
    f.write(readme_content)

print(f"Modules and migration kit written to {module_dir}")
